[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/tides/tides.ipynb)

# Tides

Tides are the difference in gravity across a body: the near side of a star cluster is pulled harder toward a mass than its far side, so as the cluster falls past, it is stretched along the line to the mass and squeezed across it, and drawn out into a stream. This example follows a disc of stars past a soft mass, a Plummer sphere whose pull fades to nothing at its centre, and compares the stars' spread with what the tides predict from the cluster's centre alone.

The tidal map at a point is a map from displacements to the planes that read the change in gravity, `Plane <- Point`: how much the pull changes for a small step. It is a sum over the masses of a squeeze along every step and a stretch along the line to each mass. Each plane it gives, multiplied by the plane that reads its step, gives the geometric derivative of gravity, `(Plane * tidal(Point)).trace(1, 2)`: its outer half, the curl, vanishes with no metric involved, and its inner half, where the metric enters, is minus four pi times the density, Poisson's equation. The cluster's shape, to first order in its size, is a map too, its deformation, driven by the tidal map at its centre, `deformation'' == tidal(deformation).dual()`, the metric turning the tidal map's reading into an acceleration. The stars' spread is a map from planes to offsets, which the deformation carries forward and its adjugate back, `deformation(spread(deformation.adjugate()))`, starting from a round cluster's radius squared times `Plane.dual()`.

In the tensor notation of Newtonian gravity the tidal map reads as the tidal tensor $T_{ij} = -\partial_i \partial_j \Phi$, Poisson's equation as $\nabla^2 \Phi = 4 \pi G \rho$, and the deformation's equation as the Jacobi equation $\ddot D = T D$, the Newtonian geodesic deviation equation.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from dataclasses import dataclass

import numpy as np
from numga import NumpyContext
from numga.algebras import VGA3D as ga

from examples.util.animation import show_animation
from examples.mechanics.tides import render
from examples.mechanics.tides.core import grid

np.set_printoptions(precision=4, suppress=True)

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
# Positions, displacements, velocities and accelerations are antivectors with intrinsic orientation; the
# planes that read them are vectors, with extrinsic orientation.
Point = ga.gatype.antivector()
Plane = ga.gatype.vector()
Even = ga.gatype.even()
Tidal = ga.gatype((Plane, Point))                           # Plane+ <- Point-
Deformation = ga.gatype((Point, Point))                     # Point- <- Point-
Spread = ga.gatype((Point, Plane))                          # Point- <- Plane+
Form = ga.gatype((Scalar, Point, Point))                    # Scalar+ <- (Point-, Point-)

## 1. A soft mass and its tidal map

A Plummer sphere pulls as a point mass beyond its core and smoothly inside it. Its acceleration at a point is a sum over the masses, and so is its tidal map: a step away from a mass weakens the pull along every direction, a squeeze, and a step toward it, read by the plane across the line to the mass, strengthens the pull along that line, a stretch. The tidal map gives the change as the plane that reads it.

In [ ]:
@dataclass(frozen=True)
class Masses:
    centres: Point                                           # [masses] Point-
    # The mass of each.
    masses: np.ndarray                                       # [masses] float
    core_radius: float

    def acceleration(self, points: Point) -> Point:
        """The pull of the masses at each point."""
        towards = self.centres - points[..., None]                           # [..., masses] Point-
        pull, _ = self._profile(towards)                                     # [..., masses] Scalar+
        return (pull * towards).sum(axis=-1)                                 # [...] Point-

    def tidal(self, points: Point) -> Tidal:
        """The tidal map at each point: the change in gravity for a small displacement, as the plane that
        reads it."""
        towards = self.centres - points[..., None]                           # [..., masses] Point-
        pull, pull_change = self._profile(towards)                           # [..., masses] Scalar+, Scalar+
        # The planes across the line to each mass, which read a step's part towards it.
        across = towards.dual_inverse()                                      # [..., masses] Plane+
        # A displacement moves away from each mass by itself, weakening the pull along every direction,
        # and changes the pull by its reading by the plane across the line to the mass, along that line.
        squeeze = -pull * Point.dual_inverse()                               # [..., masses] Plane+ <- Point-
        stretch = -2 * pull_change * across * (across & Point)               # [..., masses] Plane+ <- Point-
        return (squeeze + stretch).sum(axis=-1)                              # [...] Plane+ <- Point-

    def _profile(self, towards: Point) -> tuple[Scalar, Scalar]:
        """The pull per unit distance towards each mass, and its change with the squared distance."""
        softened = towards.scalar_norm_squared() + self.core_radius**2       # [..., masses] Scalar+
        pull = self.masses / (softened * softened.square_root())             # [..., masses] Scalar+
        return pull, -1.5 * pull / softened

## 2. Poisson's equation from the tidal map

The geometric derivative of gravity on a grid in the plane through the mass: its scalar part, the inner half, divided by minus four pi, is the density, and its curl, the outer half, is zero to round-off.

In [ ]:
# The mass and its core radius; the grid's half extents and its samples in the plane z = 0.
mass, mass_core = 1.0, 0.3
half_width, half_height, n_columns, n_rows = 3.6, 2.4, 360, 240

In [ ]:
def derivative(tides: Tidal) -> Even:
    """The geometric derivative of gravity: its inner half, where the metric enters, minus four pi times
    the density, and its outer half the curl, which needs none."""
    # The trace pairs each open plane with the displacement the tidal map reads, plane against point.
    return (Plane * tides(Point)).trace(1, 2)                               # [...] Even+


masses = Masses(mv.vector([[0.0, 0.0, 0.0]]).dual(), np.array([mass]), mass_core)
points = grid(half_width, half_height, n_columns, n_rows)                       # [rows, columns] Point-
gravity_derivative = derivative(masses.tidal(points))                           # [rows, columns] Even+
density = -gravity_derivative.select[0] / (4 * np.pi)                           # [rows, columns] Scalar+

In [ ]:
print("the tidal map a unit step from the mass, from displacements to planes:\n", masses.tidal(mv.x.dual()).coefficient_table())
print("the largest curl on the grid:", np.abs(gravity_derivative.select[2].kernel).max())

## 3. The cluster and its shape

The stars fall each on their own, by velocity Verlet. Alongside them, the cluster's centre falls the same way, and its deformation, starting as the identity map, is carried by the tidal map at the centre: its rate of change changes by the tidal map applied to it, turned by the metric into an acceleration. The stars' spread about their centre of mass is their moment, a map from planes to offsets, each offset weighted by its reading by the plane.

In [ ]:
# Annotations that name a class inside its own body wait until the class exists.
from __future__ import annotations


@dataclass(frozen=True)
class Stars:
    positions: Point                                         # [..., stars] Point-
    velocities: Point                                        # [..., stars] Point-

    def centre(self) -> Point:
        """The stars' centre of mass."""
        return self.positions.mean(axis=-1)                                  # [...] Point-

    def moment(self) -> Spread:
        """The stars' spread about their centre of mass, a map from planes to offsets: the mean of each
        offset weighted by its reading by the plane."""
        offsets = self.positions - self.centre()[..., None]                # [..., stars] Point-
        return (offsets * (Plane & offsets)).mean(axis=-1)                  # [...] Point- <- Plane+

    def fall(self, masses: Masses, dt: float) -> Stars:
        """The stars moved for a time `dt` by the velocity Verlet rule."""
        start = masses.acceleration(self.positions)                          # [..., stars] Point-
        positions = self.positions + self.velocities * dt + start * (dt**2 / 2)
        end = masses.acceleration(positions)                                 # [..., stars] Point-
        return Stars(positions, self.velocities + (start + end) * (dt / 2))


@dataclass(frozen=True)
class Shape:
    """The cluster's centre and its deformation: how an offset from the centre at the start has moved."""
    centre: Point                                            # [] Point-
    velocity: Point                                          # [] Point-
    deformation: Deformation                                 # [] Point- <- Point-
    rate: Deformation                                        # [] Point- <- Point-

    def deform(self, masses: Masses, dt: float) -> Shape:
        """The centre moved and the deformation carried by the tidal map at it, by the velocity Verlet
        rule: the deformation's rate of change changes by the tidal map applied to the deformation."""
        # The metric turns the tidal map's reading into the change in acceleration.
        start = masses.acceleration(self.centre)                             # [] Point-
        start_tide = masses.tidal(self.centre)(self.deformation).dual()      # [] Point- <- Point-
        centre = self.centre + self.velocity * dt + start * (dt**2 / 2)
        deformation = self.deformation + self.rate * dt + start_tide * (dt**2 / 2)
        end = masses.acceleration(centre)                                    # [] Point-
        end_tide = masses.tidal(centre)(deformation).dual()                  # [] Point- <- Point-
        return Shape(centre, self.velocity + (start + end) * (dt / 2),
                     deformation, self.rate + (start_tide + end_tide) * (dt / 2))

    def spread(self, radius: float) -> Spread:
        """Where a round cluster of the given starting radius has spread to, to first order in its size:
        the deformation carries the offsets forward and its adjugate carries the planes back. Its rim's
        offsets `x` have `spread.solve(x) & x == 1`."""
        # Round is where the metric enters: each plane reads the offsets across it, out to the radius.
        start = radius**2 * Plane.dual()                                     # [] Spread
        return self.deformation(start(self.deformation.adjugate()))          # [] Point- <- Plane+

    def rim(self, radius: float) -> Form:
        """The rim of a round cluster of the given starting radius, carried by the deformation: the form
        that is one, on the same offset twice, at the rim's offsets from the centre."""
        return self.spread(radius).solve(Point) & Point                      # [] Scalar+ <- (Point-, Point-)

## 4. The flyby

A uniform disc of stars falls past the mass. The animation shows the density on a log scale, the stars, and the rim the deformation predicts around their centre of mass: the cluster stretches into a stream along its path and the predicted rim follows it.

In [ ]:
# The cluster's stars, its radius and the seed that places them, its starting centre and velocity; the time
# step, the steps per frame, the frames, and a frame's duration in milliseconds.
n_stars, cluster_radius, seed = 300, 0.25, 1
start, start_velocity = [-3.0, 0.5, 0.0], [0.5, 0.0, 0.0]
dt, n_substeps, n_frames, frame_duration = 0.005, 8, 150, 40

In [ ]:
rng = np.random.default_rng(seed)
# Uniform over the disc: a radius growing as the square root, turned by a rotor to a random angle.
radii = cluster_radius * np.sqrt(rng.uniform(size=n_stars))
turns = (mv.xy * (-np.pi * rng.uniform(size=n_stars))).exp()                    # [stars] Rotor+
offsets = turns >> (mv.x.dual() * radii)                                       # [stars] Point-
centre, velocity = mv.vector(start).dual(), mv.vector(start_velocity).dual()   # [] Point-, Point-
stars = Stars(centre + offsets, velocity.broadcast_to(offsets.shape))
shape = Shape(centre, velocity, Point, 0 * Point)


def flyby(stars: Stars, shape: Shape) -> Iterator[tuple[Stars, Shape]]:
    """The stars, and the cluster's shape as the tidal map deforms it, frame by frame."""
    for _ in range(n_frames):
        yield stars, shape
        for _ in range(n_substeps):
            stars, shape = stars.fall(masses, dt), shape.deform(masses, dt)


states = list(flyby(stars, shape))
drawn = ((state_stars, state_stars.centre(), state_shape.rim(cluster_radius)) for state_stars, state_shape in states)
show_animation(render.animate(points, density, drawn), "tides_flyby", frame_duration)

In [ ]:
# checks
# The geometric derivative of gravity is minus four pi times the Plummer density, and has no curl. Through the whole
# flyby the stars spread in the plane as the deformation predicts: a uniform disc's moment is a quarter of its
# radius squared, to within the sampling of the disc and the passage through the mass's core.
towards = masses.centres - points[..., None]
softened = towards.scalar_norm_squared() + mass_core**2
plummer = (3 * masses.masses * mass_core**2 / (4 * np.pi) / (softened * softened * softened.square_root())).sum(axis=-1)
np.testing.assert_allclose((gravity_derivative + 4 * np.pi * plummer).kernel, 0.0, atol=1e-10)
planes = (mv.xy * (-np.pi * np.arange(8) / 8)).exp() >> mv.x                 # [planes] Plane+
for state_stars, state_shape in states:
    measured = (planes & state_stars.moment()(planes)).to_array()
    predicted = (planes & (state_shape.spread(cluster_radius) / 4)(planes)).to_array()
    np.testing.assert_allclose(measured, predicted, rtol=0, atol=0.3 * predicted.max())